# Bibliometry of an institution with OpenAlex — all in one (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/celoy/bibliometry/blob/main/notebooks/colab_bibliometry.ipynb)

This notebook downloads from [OpenAlex](https://openalex.org) all publications of an institution over a period, matches a list of people to their OpenAlex profiles, downloads their publications too, and produces tables, an Excel workbook and a first analysis.

**Run the cells from top to bottom** (▶ on each cell, or *Runtime → Run all*). Everything is saved in your Google Drive, in the folder **`My Drive/bibliometry`**, so you can close the notebook and come back later: each step resumes where it stopped.

### Before you start: your OpenAlex API key (2 minutes, once)

1. Create a free account at [openalex.org](https://openalex.org) (*Sign up*, top right).
2. Open [openalex.org/settings/api](https://openalex.org/settings/api) and copy your **API key**.
3. In this notebook, click the **🔑 key icon** in the left sidebar (*Secrets*) → **Add new secret**:
   * Name: `OPENALEX_API_KEY`
   * Value: paste your key
   * switch on **Notebook access**.

The key is stored in your Google account, not in the notebook, so sharing the notebook does not share your key. Without a key the pipeline still runs, but on a budget about ten times smaller.

## 1 · Install and connect your Google Drive

Google asks for permission to access your Drive: accept, so that results are kept.

In [ ]:
!pip install -q "git+https://github.com/celoy/bibliometry"

from bibliometry.env import project_root

ROOT = project_root()  # My Drive/bibliometry

## 2 · Your settings

Fill in the fields on the right, then run the cell.

* **INSTITUTION**: a name (e.g. *University of Edinburgh*, *Universidad de Chile*), an OpenAlex id (`I…`) or a ROR id. The cell lists the institutions found; if the wrong one is selected, change **INSTITUTION_CHOICE** and run again.
* **ROSTER**: the list of people (CSV or Excel), either
  * a link (also a Google Sheets *share* link, readable by anyone with the link), or
  * a file in your Drive folder `My Drive/bibliometry` (type its name, e.g. `people.xlsx`), or
  * leave it empty and tick **UPLOAD_FROM_MY_COMPUTER**.

  The list needs a column of last names (`Nom`, `Last name`…); first names, ORCIDs (`ORCID`) and labs (`Laboratoire`, `Lab`) make the matching more reliable.

In [ ]:
EMAIL = ""  # @param {type:"string"}
INSTITUTION = ""  # @param {type:"string"}
INSTITUTION_CHOICE = 1  # @param {type:"integer"}
ROSTER = ""  # @param {type:"string"}
UPLOAD_FROM_MY_COMPUTER = False  # @param {type:"boolean"}
SHEETS = ""  # @param {type:"string"}
FROM_DATE = "2020-01-01"  # @param {type:"date"}
TO_DATE = "2024-12-31"  # @param {type:"date"}

from bibliometry import pipeline
from bibliometry.config import load_api_key
from bibliometry.env import upload_roster
from bibliometry.institution import describe_institution, find_institutions
from bibliometry.openalex import OpenAlexClient, institution_works_filter

api_key = load_api_key(ROOT)
print("API key:", "found" if api_key else "NOT FOUND — see 'Before you start' above")
if UPLOAD_FROM_MY_COMPUTER:
    ROSTER = upload_roster(ROOT) or ROSTER

with OpenAlexClient(mailto=EMAIL, api_key=api_key) as c:
    candidates = find_institutions(c, INSTITUTION)
    for i, record in enumerate(candidates, start=1):
        print(("→ " if i == INSTITUTION_CHOICE else "  ") + f"{i}. {describe_institution(record)}")
    record = candidates[INSTITUTION_CHOICE - 1]
    config = pipeline.new_config(
        EMAIL, record, ROSTER, FROM_DATE, TO_DATE, ROOT,
        sheets=[s.strip() for s in SHEETS.split(",") if s.strip()],
    )
    config.validate()
    config.save(ROOT / "config.yaml")
    n = c.count("/works", institution_works_filter(config.institution.id, FROM_DATE, TO_DATE))
    print(f"\nSaved. {config.institution.name}: {n:,} works from {FROM_DATE} to {TO_DATE}.")
    print(f"OpenAlex: {c.budget}")

## 3 · The list of people

Check the counts per sheet. If ORCIDs or labs are missing although they are in your file, the column name was not recognised: rename it (`ORCID`, `Laboratoire`, `Lab`…) and run the previous cell again.

In [ ]:
import pandas as pd

from bibliometry.roster import summarise_roster

pd.set_option("display.max_colwidth", 60)
roster = pipeline.load_roster(config)
summarise_roster(roster)

## 4 · All publications of the institution

About a minute per 5,000 publications. If it is interrupted, run the cell again: it resumes.

In [ ]:
client = pipeline.make_client(config)
pipeline.harvest_institution_step(config, client)

## 5 · Matching people to their OpenAlex profiles

The same name can belong to several people, and one person is sometimes split over several OpenAlex profiles. Matches with **high** confidence (same ORCID, or same full name with publications signed with the institution) are accepted automatically; the others are listed for you to check.

In [ ]:
matches, people = pipeline.match_step(config, client)
people.match_status.value_counts()

In [ ]:
to_review = matches[~matches.accepted & (matches.verdict == "")]
to_review[["person_id", "author_id", "author_name", "confidence", "orcid_status",
           "works_in_period_with_institution", "last_known_institution", "top_topics",
           "sample_titles", "openalex_url"]]

### Your verdicts

For each row above, click its `openalex_url` to see the profile, then write `"ok"` (it is this person) or `"wrong"` (somebody else) below, and run the cell. You can come back to this later: verdicts are saved in your Drive.

In [ ]:
verdicts = {
    # ("person_id", "author_id"): "ok",
    # ("person_id", "author_id"): "wrong",
}
if verdicts:
    pipeline.set_verdicts(config, verdicts)
matches, people = pipeline.match_step(config, None)  # applies verdicts, no download
people.match_status.value_counts()

People not found are usually people who did not publish in the period. If you know someone's OpenAlex id, add an `OpenAlex` column to your list and run from step 2 again.

In [ ]:
people[people.match_status == "not found"][["person_id", "lab", "orcid", "sheet"]]

## 6 · Publications of the people, tables and Excel workbook

All publications of the accepted profiles are downloaded, including those not signed with the institution (kept, but flagged).

In [ ]:
pipeline.harvest_people_step(config, client)
tables = pipeline.tables_step(config)
{name: len(frame) for name, frame in tables.items()}

In [ ]:
from bibliometry.env import offer_download

excel = sorted((ROOT / "data" / "exports").glob("*.xlsx"))[-1]
offer_download(excel)  # also in My Drive/bibliometry/data/exports

## 7 · Lists, most-cited shares and venues

### The two publication lists

* **List 1 — institution**: every publication of the period with at least one author signing with the institution.
* **List 2 — people of the list**: publications where a person of your list signed **as a member of the institution** (a subset of list 1; what they published under another affiliation is not counted).

Both are saved with **every OpenAlex field** in `data/lists/` (`list1_institution.xlsx`, `list2_roster.xlsx`, and `.parquet` for Python), together with the venues (`venues.xlsx`) and the indicators below (`indicators.xlsx`).

The first run also downloads a **world reference** of all active journals (~64,000, a few minutes, once).

In [ ]:
lists_out = pipeline.lists_step(config, pipeline.make_client(config))

### Top 10% and 1% most cited

OpenAlex compares each publication with publications of the same field, year and type (`citation_normalized_percentile`). For an average institution, 10% of its publications are in the world's top 10%, and 1% in the top 1%. Shares are computed over publications that have a percentile (`coverage`); recent years are less reliable, as citations are still accumulating.

Two scopes: **all works**, and **research, deduplicated** (articles, reviews, conference papers, chapters, books, preprints, theses, reports; a paper and its preprint counted once).

In [ ]:
cited = lists_out["top_cited"]
cited.style.format({"coverage": "{:.0%}", "share_top10": "{:.1%}", "share_top1": "{:.1%}"})

#### The top-1% papers

Research outputs, deduplicated. In `authors`, `*` marks authors signing with the institution and `[person-id]` the people of your list.

In [ ]:
lists_out["top1_papers_list2"]

In [ ]:
lists_out["top1_papers_list1"]

### Venues

Each publication's venue is its journal (or conference, book series…); a HAL/arXiv deposit is replaced by the journal when OpenAlex knows both.

**Headline metric — 2-year mean citedness** (OpenAlex's analogue of the impact factor: mean citations received in a year by the journal's papers of the two previous years). Two flags:

* `world_top10_2yr_mean_citedness` — the journal is in the top 10% of **all active journals in the world of the same main subfield** (`world_pct_…` = share of those journals doing better; `world_n` = their number). This compares like with like;
* `top10_2yr_mean_citedness` — in the top 10% of the journals used in list 1 (all fields together).

`broad_scope` marks journals spanning several domains (Scientific Reports, PLOS ONE, Nature…), whose main subfield is somewhat arbitrary.

**Secondary metrics — h-index and i10-index** (further right in the tables): they grow with a journal's size and age, so they say as much about volume as about impact.

In [ ]:
venues_table = lists_out["venues"]
print(f"{venues_table.attrs.get('n_ranked', venues_table.ranked.sum())} journals ranked")
venues_table[venues_table.ranked][
    ["venue_name", "main_subfield", "broad_scope", "works_list1", "works_list2",
     "2yr_mean_citedness", "world_pct_2yr_mean_citedness", "world_top10_2yr_mean_citedness",
     "top10_2yr_mean_citedness", "world_n"]
].head(30)

Share of publications (in ranked journals) that appeared in a top-10% journal — 2-year citedness first, then the secondary metrics:

In [ ]:
lists_out["top_venues"].T

## 8 · First analysis

Scope: research publications with at least one author signing with the institution; a paper and its preprint / HAL copy are counted once.

In [ ]:
from bibliometry import analysis as A

works_all, people = tables["works"], tables["people"]
works = A.deduplicate(A.in_scope(works_all))
print(f"{len(works):,} publications in scope out of {len(works_all):,} downloaded")
fig = A.plot_lines(A.works_per_year(works), "Publications per year")

In [ ]:
fig = A.plot_lines(A.works_per_lab(works, people), "Publications per year and lab (people of the list)")

In [ ]:
A.roster_coverage(works, works_all)

In [ ]:
fig = A.plot_bars(A.type_counts(works), "Publication types")

In [ ]:
fig = A.plot_stacked_shares(A.open_access(works), "Open-access status (share of publications)")

In [ ]:
A.top_venues(works[works.source_type != "repository"], 20)

In [ ]:
A.citation_summary(works).round(2)

In [ ]:
fig = A.plot_bars(A.partner_countries(tables["authorships"], works, 20), "Countries of co-signing institutions")

In [ ]:
A.partners(tables["authorships"], works, config.institution.id, 25)

In [ ]:
fig = A.plot_bars(A.top_topics(tables["topics"], works, "field", 15), "OpenAlex fields (primary topic)")